# E4b - SiLU rerun with extended Phase 2 (revision experiment)

Companion to `E4_Activation_Homogeneity.ipynb`. The first SiLU run used
`phase2=500` and produced plateauing NC1 just above 0.05 for 2/3 seeds
(min NC1 = 0.0585, 0.0524, 0.0498). The fn values were in 2.1-3.3, which
is directionally consistent with H-M1 (smooth non-homogeneous activations
land near GELU's fn* = 2.13) but did not yield a clean fn at NC1 < 0.05
with 3-seed CV.

This rerun keeps everything identical except `phase2 = 1000` (and matches
LeakyReLU's per-run budget given its faster collapse), so all three seeds
have a fair shot at reaching NC1 < 0.05.

Outputs are written to `actSiLU_extended_s{0,1,2}.csv` so the original
E4 SiLU CSVs are not overwritten.

Compute: ~1 h on T4 / ~30 min on A100.

Same Phase-2-only T_NC convention and same heartbeat/seeding as E4.


In [1]:

import torch, torchvision, time, os, sys
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Runs on both Kaggle and Colab. Output dir is auto-detected.
if os.path.isdir('/kaggle/working'):
    PLATFORM = 'kaggle'
    SAVE_DIR = '/kaggle/working/'
    DATA_DIR = '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    SAVE_DIR = '/content/'
    DATA_DIR = '/content/data/'
else:
    PLATFORM = 'local'
    SAVE_DIR = './'
    DATA_DIR = './data/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), \
    'No GPU - Runtime -> Change runtime type -> A100 (Colab) or enable GPU (Kaggle)'
print(f'Platform: {PLATFORM}')
print(f'SAVE_DIR: {SAVE_DIR}')
print(f'GPU:      {torch.cuda.get_device_name(0)}')
print(f'Torch:    {torch.__version__}')

Platform: colab
SAVE_DIR: /content/
GPU:      Tesla T4
Torch:    2.10.0+cu128


In [2]:

transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
trainset = torchvision.datasets.MNIST(DATA_DIR,
    train=True,  download=True, transform=transform)
testset  = torchvision.datasets.MNIST(DATA_DIR,
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.62MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 155kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.8MB/s]

MNIST: 60,000 train / 10,000 test


In [3]:

class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU,
                 num_classes=10, in_dim=784):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(in_dim, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

print('MLP defined.')

MLP defined.


In [4]:

@torch.no_grad()
def compute_nc(model, loader, K):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c])
               for c in range(K)) / len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask] - (-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1 - (Mn * Wn).sum(1).mean()).item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3,
            'feat_norm': H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

print('NC metrics + evaluate ready.')

NC metrics + evaluate ready.


In [5]:

def run_twophase(model, name, lr=1e-3, wd=1e-4,
                 phase1=200, phase2=1000, nc_every=10, K=10):
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception:
        pass
    model = model.to(DEVICE)
    rows = []; terminal = False
    t_nc_strict = None; fn_at_strict = None
    t_nc_relaxed = None; fn_at_relaxed = None
    t0 = time.time()
    for phase, loss_fn, n_ep in [(1, 'ce', phase1), (2, 'mse', phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_fn == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y, K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sch.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep}')
                if not terminal and ep_l % (nc_every * 5) == 0:
                    print(f'  [{name}] ep={ep} tr={tr:.4f} te={te:.4f} '
                          f'(pre-terminal) t={(time.time()-t0)/60:.1f}m')
                if terminal:
                    nc = compute_nc(model, train_loader, K)
                else:
                    nc = {'nc1': None, 'nc2': None,
                          'nc3': None, 'feat_norm': None}
                rows.append({'epoch': ep, 'phase': phase,
                             'train': tr, 'test': te, **nc})
                # T_NC is defined as the first Phase-2 epoch with NC1
                # below the threshold. Phase-1 NC1 dips (seen e.g. with
                # Tanh) are transient and do not reflect equilibrium
                # collapse; the published numbers use the same definition.
                if nc['nc1'] is not None and phase == 2:
                    if t_nc_relaxed is None and nc['nc1'] < 0.05:
                        t_nc_relaxed = ep; fn_at_relaxed = nc['feat_norm']
                        print(f'  [{name}] NC1<0.05 at ep {ep} '
                              f'fn={fn_at_relaxed:.4f}')
                    if t_nc_strict is None and nc['nc1'] < 0.01:
                        t_nc_strict = ep; fn_at_strict = nc['feat_norm']
                        print(f'  [{name}] NC1<0.01 at ep {ep} '
                              f'fn={fn_at_strict:.4f}')
    print(f'  [{name}] Done in {(time.time()-t0)/60:.1f} min')
    return (pd.DataFrame(rows),
            t_nc_strict, fn_at_strict,
            t_nc_relaxed, fn_at_relaxed)

print('run_twophase ready.')

run_twophase ready.


In [6]:

ACTIVATIONS = [
    ('SiLU_extended', lambda: nn.SiLU()),
]

results = []
for act_name, act_ctor in ACTIVATIONS:
    for seed in range(3):
        print(f'\n=== Activation={act_name}  seed={seed} ===')
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        model = MLP(depth=5, width=512, act_cls=act_ctor,
                    num_classes=10, in_dim=784)
        df, ts, fs, tr_, fr = run_twophase(model,
                                           f'act{act_name}-s{seed}',
                                           lr=1e-3, wd=1e-4,
                                           phase1=200, phase2=1000)
        df.to_csv(f'{SAVE_DIR}act{act_name}_s{seed}.csv', index=False)
        results.append({'activation': act_name, 'seed': seed,
                        'T_NC_strict': ts, 'fn_strict': fs,
                        'T_NC_relaxed': tr_, 'fn_relaxed': fr,
                        'test_acc_final': df.test.iloc[-1]})
        pd.DataFrame(results).to_csv(
            f'{SAVE_DIR}silu_extended_summary.csv', index=False)
        print(f'  saved act{act_name}_s{seed}.csv and summary')

summary = pd.DataFrame(results)
print('\n=== Activation probe summary ===')
print(summary.to_string(index=False))

print('\nReference values from published paper:')
print('  ReLU  fn*=1.077 (N=3, NC1<0.01)')
print('  GELU  fn*=2.128 (N=3, NC1<0.05)')
print('  Tanh  fn*=1.325 (N=3, NC1<0.05)')

for act in ('SiLU_extended',):
    sub = summary[(summary.activation == act)]
    if not sub.empty:
        for col, label in [('fn_strict', 'NC1<0.01'),
                           ('fn_relaxed', 'NC1<0.05')]:
            ok = sub.dropna(subset=[col])
            if len(ok) >= 2:
                fns = ok[col].values
                print(f'  {act:>9} {label}: mean={fns.mean():.4f} '
                      f'std={fns.std():.4f} N={len(fns)}')


=== Activation=SiLU_extended  seed=0 ===


W0526 13:01:17.076000 1007 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  [actSiLU_extended-s0] Terminal phase at epoch 10
  [actSiLU_extended-s0] NC1<0.05 at ep 690 fn=3.2850
  [actSiLU_extended-s0] Done in 80.9 min
  saved actSiLU_extended_s0.csv and summary

=== Activation=SiLU_extended  seed=1 ===
  [actSiLU_extended-s1] Terminal phase at epoch 10
  [actSiLU_extended-s1] Done in 81.4 min
  saved actSiLU_extended_s1.csv and summary

=== Activation=SiLU_extended  seed=2 ===
  [actSiLU_extended-s2] Terminal phase at epoch 10
  [actSiLU_extended-s2] NC1<0.05 at ep 340 fn=2.2405
  [actSiLU_extended-s2] Done in 82.1 min
  saved actSiLU_extended_s2.csv and summary

=== Activation probe summary ===
   activation  seed T_NC_strict fn_strict  T_NC_relaxed  fn_relaxed  test_acc_final
SiLU_extended     0        None      None         690.0    3.284964          0.9751
SiLU_extended     1        None      None           NaN         NaN          0.9764
SiLU_extended     2        None      None         340.0    2.240484          0.9766

Reference values from published

In [7]:
out_files = [f'{SAVE_DIR}silu_extended_summary.csv']
for seed in range(3):
    out_files.append(f'{SAVE_DIR}actSiLU_extended_s{seed}.csv')
if PLATFORM == 'colab':
    from google.colab import files
    for fp in out_files:
        if os.path.exists(fp):
            files.download(fp)
else:
    print('Files saved in', SAVE_DIR)
    for fp in out_files:
        if os.path.exists(fp):
            print(' ', fp, f'({os.path.getsize(fp)} bytes)')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>